# 01 — Bronze → Silver: CMS Physician Provider & Service

Reads the raw CMS Medicare Physician & Other Practitioners (by Provider and
Service) JSON files from `bronze/`, applies explicit casts, derives the
analytical columns the Gold layer needs (totals, ratios, RUCA bucket,
Provider_Tier), and writes a single Delta table to `silver/`.

Idempotent: every run does a full `mode("overwrite")` so reruns are safe.

Sample-mode widget: set to `true` to limit the read to 1,000 rows for
cost-free dry runs before a full-volume run.

In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType

from utils.secrets    import configure_adls_oauth
from utils.paths      import BRONZE_PHYSICIAN_GLOB, SILVER_PHYSICIAN
from utils.schemas    import CMS_RAW_SCHEMA, SILVER_CASTS
from utils.credentials import normalize_credentials_to_tier

spark = SparkSession.builder.appName("bronze_to_silver_physician").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
configure_adls_oauth(spark, dbutils)

DECIMAL_MONEY = DecimalType(18, 2)
DECIMAL_RATIO = DecimalType(12, 6)

## Sample-mode harness — set to "true" for dry runs on 1,000 rows.



In [0]:
dbutils.widgets.dropdown("sample_mode", "false", ["true", "false"])
SAMPLE_MODE = dbutils.widgets.get("sample_mode") == "true"
print(f"sample_mode = {SAMPLE_MODE}")

sample_mode = False


## Bronze read — every column as StringType (no inferSchema)



In [0]:
bronze = (
    spark.read
        .schema(CMS_RAW_SCHEMA)
        .option("multiLine", "true")
        .json(BRONZE_PHYSICIAN_GLOB)
)
if SAMPLE_MODE:
    bronze = bronze.limit(1000)
print(f"Bronze row count: {bronze.count():,}")

Bronze row count: 9,665,647


## Trim + null-normalize every string column



In [0]:
for c in [f.name for f in CMS_RAW_SCHEMA.fields]:
    bronze = bronze.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))),
    )

## Explicit Silver casts (replaces inferSchema)



In [0]:
for col_name, cast_type in SILVER_CASTS.items():
    if cast_type in ("int", "long", "bigint", "short"):
        bronze = bronze.withColumn(col_name, F.col(col_name).cast("double").cast(cast_type))
    else:
        bronze = bronze.withColumn(col_name, F.col(col_name).cast(cast_type))

## Derived columns: totals, ratios, flags, Provider_Tier



In [0]:
silver = (
    bronze
    .filter(F.col("Rndrng_Prvdr_Cntry") == "US")
    .withColumn(
        "service_sk",
        F.sha2(
            F.concat_ws("||", F.col("Rndrng_NPI"), F.col("HCPCS_Cd"), F.col("Place_Of_Srvc")),
            256,
        ),
    )
    # Totals — DECIMAL(18,2) avoids float drift on Avg × Tot_Srvcs
    .withColumn("Tot_Sbmtd_Chrg",
        (F.col("Avg_Sbmtd_Chrg")     * F.col("Tot_Srvcs")).cast(DECIMAL_MONEY))
    .withColumn("Tot_Mdcr_Alowd_Amt",
        (F.col("Avg_Mdcr_Alowd_Amt") * F.col("Tot_Srvcs")).cast(DECIMAL_MONEY))
    .withColumn("Tot_Mdcr_Pymt_Amt",
        (F.col("Avg_Mdcr_Pymt_Amt")  * F.col("Tot_Srvcs")).cast(DECIMAL_MONEY))
    .withColumn("Tot_Mdcr_Stdzd_Amt",
        (F.col("Avg_Mdcr_Stdzd_Amt") * F.col("Tot_Srvcs")).cast(DECIMAL_MONEY))
    .withColumn("Geo_Premium",
        ((F.col("Avg_Mdcr_Pymt_Amt") - F.col("Avg_Mdcr_Stdzd_Amt"))
         * F.col("Tot_Srvcs")).cast(DECIMAL_MONEY))
    .withColumn("Patient_Exposure",
        ((F.col("Avg_Sbmtd_Chrg") - F.col("Avg_Mdcr_Pymt_Amt"))
         * F.col("Tot_Benes")).cast(DECIMAL_MONEY))
    # Ratios — DECIMAL(12,6), div/0 guarded
    .withColumn("Markup_Ratio",
        F.when(F.col("Avg_Mdcr_Alowd_Amt") > 0,
               (F.col("Avg_Sbmtd_Chrg") / F.col("Avg_Mdcr_Alowd_Amt"))
               .cast(DECIMAL_RATIO)))
    .withColumn("Mdcr_Pymt_Rate",
        F.when(F.col("Avg_Mdcr_Alowd_Amt") > 0,
               (F.col("Avg_Mdcr_Pymt_Amt") / F.col("Avg_Mdcr_Alowd_Amt"))
               .cast(DECIMAL_RATIO)))
    .withColumn("Geo_Adj_Factor",
        F.when(F.col("Avg_Mdcr_Stdzd_Amt") > 0,
               (F.col("Avg_Mdcr_Pymt_Amt") / F.col("Avg_Mdcr_Stdzd_Amt"))
               .cast(DECIMAL_RATIO)))
    # Boolean flags
    .withColumn("Is_Drug",          F.col("HCPCS_Drug_Ind") == F.lit("Y"))
    .withColumn("Is_Facility",      F.col("Place_Of_Srvc")   == F.lit("F"))
    .withColumn("Is_Participating", F.col("Rndrng_Prvdr_Mdcr_Prtcptg_Ind") == F.lit("Y"))
    .withColumn("Is_Org",           F.col("Rndrng_Prvdr_Ent_Cd") == F.lit("O"))
    .withColumn("RUCA_Int",         F.col("Rndrng_Prvdr_RUCA").cast("double").cast("int"))
    .withColumn("Is_Rural",         F.col("RUCA_Int").between(4, 10))
    .withColumn("RUCA_Bucket",
        F.when(F.col("RUCA_Int").between(1, 3),  F.lit("Urban"))
         .when(F.col("RUCA_Int").between(4, 6),  F.lit("Large rural"))
         .when(F.col("RUCA_Int").between(7, 10), F.lit("Small rural / isolated"))
         .otherwise(F.lit("Unknown")))
    .withColumn("Provider_Tier", normalize_credentials_to_tier(F.col("Rndrng_Prvdr_Crdntls")))
    .withColumn("Source_Year", F.lit(2023))
    .withColumn("Ingested_At", F.current_timestamp())
)

## Strict idempotent write — full overwrite, deterministic on rerun



In [0]:
(silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("Rndrng_Prvdr_State_Abrvtn")
    .save(SILVER_PHYSICIAN))

print(f"Silver written → {SILVER_PHYSICIAN}")

Silver written → abfss://silver@sthealthcareplatdev.dfs.core.windows.net/physician_by_provider_service/


## Optimize + Z-ORDER to keep Power BI Import-mode fast



In [0]:
spark.sql(
    f"OPTIMIZE delta.`{SILVER_PHYSICIAN}` ZORDER BY (HCPCS_Cd, Rndrng_Prvdr_Type)"
)
print("Silver OPTIMIZE complete.")

Silver OPTIMIZE complete.
